# SRQ-FLY self-contained final study
This notebook performs train-only nested validation, locks the selected hyperparameters, refits analytic learners on the complete official training split, evaluates six fresh replicates on CIFAR-100, CUB-200-2011 and the disclosed legacy ImageNet-R split, and exports publication-style plots. Run cells in order; never edit the search grid after test extraction.

In [ ]:
# === Edit repository/path values only. Do not edit protocol seeds, grid or method settings. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'paper/srq-fly-draft'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_ROOT = '/content/srq_selfcontained_features'
SELECTION_WTA_ROOT = '/content/srq_selection_wta'
FINAL_WTA_ROOT = '/content/srq_final_wta'
SELECTION_ROOT = '/content/srq_selfcontained_selection'
OUTPUT_ROOT = '/content/srq_selfcontained_results'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_PROTOCOL_SHA256 = 'e1c9fefedf9a55beb4d7734f8dcd8fafdcd551bd76af8a703d27b0b447e16759'
EXPECTED_RUNNER_SHA256 = '92ebd8ec12c73ac7da9ee1f591e8613d88515418db15a2918db13d1c98fe5cdc'

In [ ]:
# Fresh clone, dependency install, GPU check and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
PROTOCOL = 'configs/srq_fly_selfcontained_final.json'
RUNNER = 'tools/srq_fly_selfcontained.py'
assert sha(PROTOCOL) == EXPECTED_PROTOCOL_SHA256
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Repository must start clean.'
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('protocol:', EXPECTED_PROTOCOL_SHA256)
print('runner:', EXPECTED_RUNNER_SHA256)

In [ ]:
# Download the exact frozen ViT checkpoint and three processed datasets.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOTS = {
  'cifar100': kagglehub.dataset_download('zaphat206/cifar-100'),
  'cub200': kagglehub.dataset_download('zaphat206/cub-200-2011'),
  'imagenetr': kagglehub.dataset_download('zaphat206/imagenet-r'),
}
print('checkpoint:', CHECKPOINT_PATH)
print(json.dumps(DATASET_ROOTS, indent=2))

In [ ]:
# Audit dataset identity without extracting held-out features.
CUB_AUDIT = '/content/cub_selfcontained_audit.json'
IMAGENETR_AUDIT = '/content/imagenetr_selfcontained_audit.json'
cub = subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca'])
assert cub.returncode == 0
imagenetr = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert imagenetr.returncode == 2, 'Expected locked legacy-overlap disclosure.'
audit = json.loads(Path(IMAGENETR_AUDIT).read_text())
assert audit['cross_split_duplicate_content_count'] == 19
assert audit['cross_split_conflicting_label_duplicate_count'] == 18
print('DATASET AUDIT PASS; ImageNet-R will be labeled legacy processed split.')

In [ ]:
# Extract frozen TRAIN features only. test.pt must remain absent.
protocol = json.loads(Path(PROTOCOL).read_text())
Path(FEATURE_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg = protocol['datasets'][key]; cache = Path(FEATURE_CACHE_ROOT)/key
    if not (cache/'train.pt').is_file():
        command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only',
          '--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,
          '--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],
          '--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_{key}',
          '--dataset',cfg['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit',
          '--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),
          '--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
        print(f'TRAIN EXTRACT START {key}', flush=True); subprocess.run(command, check=True)
    assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
    print(f'TRAIN CACHE READY {key}; test.pt absent')
print('ALL TRAIN-ONLY FEATURE CACHES READY')

In [ ]:
# Synthetic correctness, split-leakage and state tests. No real test feature is opened.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_selfcontained.py','tests/test_srq_fly_math.py','tests/test_srq_fly_learner.py'], check=True)
print('SELF-CONTAINED CORRECTNESS GATE: PASS')

In [ ]:
# TRAIN-ONLY GRID SEARCH. Three development replicates; test.pt is physically absent.
audit_paths = {'cifar100': None, 'cub200': CUB_AUDIT, 'imagenetr': IMAGENETR_AUDIT}
for key in ('cifar100','cub200','imagenetr'):
    command = [sys.executable,'-u',RUNNER,'select','--protocol',PROTOCOL,'--dataset-key',key,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--code-cache-root',SELECTION_WTA_ROOT,
      '--output-root',SELECTION_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'GRID SEARCH START {key}: 12 lambdas x 3 development replicates', flush=True)
    subprocess.run(command, check=True)
    selected = json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    print(key, selected['status'], 'FLY/SRQ lambda=',selected['selected_fly_family_lambda'],'raw lambda=',selected['selected_raw_ridge_lambda'])
    assert selected['status'] == 'SELECTION_COMPLETE', 'Grid endpoint selected; stop before test and review.'
    assert selected['uses_test_set'] is False
print('TRAIN-ONLY SELECTION COMPLETE FOR ALL DATASETS')

In [ ]:
# Display validation evidence, then create the immutable single-use lock.
import pandas as pd
selection_rows=[]
for key in ('cifar100','cub200','imagenetr'):
    result=json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    selection_rows.append({'dataset':key,'FLY/SRQ lambda':result['selected_fly_family_lambda'],
      'Raw Ridge lambda':result['selected_raw_ridge_lambda'],'status':result['status']})
display(pd.DataFrame(selection_rows))
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-u',RUNNER,'lock','--protocol',PROTOCOL,'--selection-root',SELECTION_ROOT,
  '--output-root',OUTPUT_ROOT,'--require-clean-git'], check=True)
AUTHORIZATION = str(Path(OUTPUT_ROOT)/'authorization.json')
print(json.dumps(json.loads(Path(AUTHORIZATION).read_text()), indent=2))

## Irreversible test boundary
The grid, nested splits and selected lambdas are now hashed. From the next cell onward, test features may be materialized. Do not edit the protocol, grid or selected settings based on any test output. Identical interrupted units may resume only under the same authorization.

In [ ]:
# Extract HELD-OUT TEST features only after the immutable lock.
for key in ('cifar100','cub200','imagenetr'):
    command = [sys.executable,'-u',RUNNER,'extract-test','--protocol',PROTOCOL,'--dataset-key',key,
      '--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--root',DATASET_ROOTS[key],
      '--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print(f'TEST EXTRACTION START {key}', flush=True); subprocess.run(command, check=True)
print('ALL AUTHORIZED TEST FEATURE CACHES READY')

In [ ]:
# Helper: refit each learner from empty state on full train, then evaluate six fresh replicates.
def run_final(key):
    command=[sys.executable,'-u',RUNNER,'evaluate','--protocol',PROTOCOL,'--dataset-key',key,
      '--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--code-cache-root',FINAL_WTA_ROOT,
      '--output-root',OUTPUT_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'FINAL START {key}: 6 replicates x 3 methods', flush=True)
    subprocess.run(command, check=True)
    payload=json.loads(Path(OUTPUT_ROOT,key,'heldout_results.json').read_text())
    rows=[]
    for replicate in payload['seed_results']:
      for method,result in replicate['methods'].items():
        rows.append({'replicate':replicate['replicate_index'],'method':method,'status':result['status'],
          'final_accuracy':result.get('final_accuracy'),'AIA':result.get('average_incremental_accuracy'),
          'forgetting':result.get('forgetting'),'state_MiB':None if result.get('persistent_state_bytes') is None else result['persistent_state_bytes']/2**20})
    display(pd.DataFrame(rows)); print('STATUS:',payload['status'])

In [ ]:
# CIFAR-100 final evaluation.
run_final('cifar100')

In [ ]:
# CUB-200-2011 final evaluation.
run_final('cub200')

In [ ]:
# ImageNet-R legacy processed-split final evaluation.
run_final('imagenetr')

In [ ]:
# Aggregate mean, standard deviation and 95% confidence intervals.
subprocess.run([sys.executable,'-u',RUNNER,'summarize','--protocol',PROTOCOL,'--output-root',OUTPUT_ROOT], check=True)
metrics=pd.read_csv(Path(OUTPUT_ROOT)/'metrics_summary.csv')
display(metrics.sort_values(['dataset','method']))
summary=json.loads(Path(OUTPUT_ROOT,'final_summary.json').read_text())
print('Paired SRQ minus exact FLY AIA:',json.dumps(summary['paired_srq_minus_exact_fly_aia'],indent=2))
print('ImageNet-R disclosure:',summary['imagenetr_disclosure'])

In [ ]:
# Generate all reporting plots: task curves, accuracy, memory, Pareto, forgetting and paired seeds.
import matplotlib.pyplot as plt, numpy as np, seaborn as sns
sns.set_theme(style='whitegrid',context='talk')
plot_dir=Path(OUTPUT_ROOT)/'plots'; plot_dir.mkdir(parents=True,exist_ok=True)
curves=pd.read_csv(Path(OUTPUT_ROOT)/'task_curves.csv')
names={'exact_fly_10000':'Exact FLY-10k','srq_fly_10000':'SRQ-FLY-10k','raw_ridge':'Raw Ridge'}
colors={'exact_fly_10000':'#4C78A8','srq_fly_10000':'#E45756','raw_ridge':'#54A24B'}
datasets=['cifar100','cub200','imagenetr']; methods=list(names)
fig,axes=plt.subplots(1,3,figsize=(21,5.5),sharey=True)
for ax,dataset in zip(axes,datasets):
  view=curves[curves.dataset==dataset]
  for method in methods:
    group=view[view.method==method].groupby('task').average_seen_accuracy
    mean=group.mean(); ci=2.571*group.std(ddof=1)/np.sqrt(group.count())
    ax.plot(mean.index,mean.values,label=names[method],color=colors[method],marker='o',ms=3)
    ax.fill_between(mean.index,mean-ci,mean+ci,color=colors[method],alpha=.16)
  ax.set_title(dataset); ax.set_xlabel('Task'); ax.set_ylabel('Average seen-class accuracy (%)')
axes[0].legend(fontsize=10); fig.tight_layout(); fig.savefig(plot_dir/'01_accuracy_by_task.png',dpi=220,bbox_inches='tight'); plt.show()
fig,axes=plt.subplots(1,2,figsize=(15,5.5))
for ax,metric,title in zip(axes,['final_accuracy','average_incremental_accuracy'],['Final accuracy','Average incremental accuracy']):
  x=np.arange(3); width=.25
  for j,method in enumerate(methods):
    view=metrics[metrics.method==method].set_index('dataset').loc[datasets]
    y=view[f'{metric}_mean'].to_numpy(); err=np.vstack([y-view[f'{metric}_ci95_low'],view[f'{metric}_ci95_high']-y])
    ax.bar(x+(j-1)*width,y,width,label=names[method],color=colors[method],yerr=err,capsize=4)
  ax.set_xticks(x,datasets); ax.set_ylabel('Accuracy (%)'); ax.set_title(title)
axes[0].legend(fontsize=10); fig.tight_layout(); fig.savefig(plot_dir/'02_final_and_aia.png',dpi=220,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(10,5.5)); x=np.arange(3); width=.25
for j,method in enumerate(methods):
  view=metrics[metrics.method==method].set_index('dataset').loc[datasets]
  ax.bar(x+(j-1)*width,view.persistent_state_bytes_mean/2**20,width,label=names[method],color=colors[method])
ax.set_yscale('log'); ax.set_xticks(x,datasets); ax.set_ylabel('Persistent learner state (MiB, log scale)'); ax.legend(fontsize=10)
fig.tight_layout(); fig.savefig(plot_dir/'03_persistent_state.png',dpi=220,bbox_inches='tight'); plt.show()
fig,axes=plt.subplots(1,3,figsize=(18,5.5),sharey=True)
for ax,dataset in zip(axes,datasets):
  view=metrics[metrics.dataset==dataset]
  for method in methods:
    row=view[view.method==method].iloc[0]; ax.scatter(row.persistent_state_bytes_mean/2**20,row.average_incremental_accuracy_mean,s=130,color=colors[method],label=names[method])
    ax.annotate(names[method],(row.persistent_state_bytes_mean/2**20,row.average_incremental_accuracy_mean),xytext=(5,5),textcoords='offset points',fontsize=9)
  ax.set_xscale('log'); ax.set_title(dataset); ax.set_xlabel('Persistent state (MiB, log)'); ax.set_ylabel('AIA (%)')
fig.tight_layout(); fig.savefig(plot_dir/'04_accuracy_memory_pareto.png',dpi=220,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(10,5.5)); x=np.arange(3); width=.25
for j,method in enumerate(methods):
  view=metrics[metrics.method==method].set_index('dataset').loc[datasets]; ax.bar(x+(j-1)*width,view.forgetting_mean,width,label=names[method],color=colors[method])
ax.set_xticks(x,datasets); ax.set_ylabel('Forgetting (pp; lower is better)'); ax.legend(fontsize=10)
fig.tight_layout(); fig.savefig(plot_dir/'05_forgetting.png',dpi=220,bbox_inches='tight'); plt.show()
paired_rows=[]
for dataset in datasets:
  payload=json.loads(Path(OUTPUT_ROOT,dataset,'heldout_results.json').read_text())
  for item in payload['seed_results']:
    paired_rows.append({'dataset':dataset,'replicate':item['replicate_index'],'SRQ minus FLY AIA':item['methods']['srq_fly_10000']['average_incremental_accuracy']-item['methods']['exact_fly_10000']['average_incremental_accuracy']})
paired_df=pd.DataFrame(paired_rows); fig,ax=plt.subplots(figsize=(10,5.5)); sns.stripplot(data=paired_df,x='dataset',y='SRQ minus FLY AIA',jitter=False,size=9,ax=ax)
ax.axhline(0,color='black',lw=1); ax.set_ylabel('Paired SRQ-FLY − Exact FLY AIA (pp)')
fig.tight_layout(); fig.savefig(plot_dir/'06_paired_seed_differences.png',dpi=220,bbox_inches='tight'); plt.show()
print('PLOTS:',sorted(path.name for path in plot_dir.glob('*.png')))

In [ ]:
# Export compact evidence. Sample-level feature/WTA caches are deliberately excluded.
from google.colab import files
shutil.copy2(PROTOCOL,Path(OUTPUT_ROOT)/'locked_protocol.json')
shutil.copy2(RUNNER,Path(OUTPUT_ROOT)/'locked_runner.py')
selection_copy=Path(OUTPUT_ROOT)/'train_only_selection'; selection_copy.mkdir(exist_ok=True)
for key in ('cifar100','cub200','imagenetr'): shutil.copy2(Path(SELECTION_ROOT,key,'selection.json'),selection_copy/f'{key}_selection.json')
archive=shutil.make_archive('/content/srq_fly_selfcontained_three_dataset_results','zip',root_dir=OUTPUT_ROOT)
print('artifact:',archive,'size=',Path(archive).stat().st_size,'SHA-256=',sha(archive))
files.download(archive)